# GCS CSV ETL Notebook

Goal: practice a Cloud Storage style ETL workflow. This notebook runs locally with sample data and can be extended to read/write real `gs://` objects.

## Interview Story

I built a batch ETL workflow that reads raw CSV data, validates records, enriches rows, and writes a clean output file. In production this can run as a Cloud Run Job or Composer task using Cloud Storage as the landing zone.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

raw_path = DATA_DIR / 'users.csv'
df = pd.read_csv(raw_path)
df

In [ ]:
def score_band(score: int) -> str:
    if score >= 85:
        return 'high'
    if score >= 60:
        return 'medium'
    return 'low'

clean_df = df.dropna(subset=['user_id']).copy()
clean_df['name'] = clean_df['name'].str.strip()
clean_df['department'] = clean_df['department'].str.lower().str.strip()
clean_df['score_band'] = clean_df['score'].apply(score_band)
clean_df

In [ ]:
output_path = OUTPUT_DIR / 'users_clean.csv'
clean_df.to_csv(output_path, index=False)
print(f'Wrote {len(clean_df)} rows to {output_path}')

## Optional: Read from Cloud Storage

Uncomment and configure this block after running `gcloud auth application-default login`.

In [ ]:
# from google.cloud import storage
# client = storage.Client(project='YOUR_PROJECT_ID')
# bucket = client.bucket('YOUR_BUCKET')
# blob = bucket.blob('input/users.csv')
# raw_text = blob.download_as_text()
# print(raw_text[:200])